> **Note.**  
> [More Cultural Names](https://github.com/hmlendea/more-cultural-names) (MCN) is used here strictly as an **evaluation oracle**: its attested exonyms serve as a gold standard against which NTE's output is measured. MCN data has not been used to populate, enrich, or tune NTE's own lookup database, which is built independently from [GeoNames](https://www.geonames.org/) alternate-name dumps and [Wikidata](https://www.wikidata.org/) CC0 snapshots. Any agreement between NTE output and MCN entries therefore reflects genuine coverage by NTE's pipeline, not data leakage.
>
> MCN is distributed under the [GPL-3.0 License](https://github.com/hmlendea/more-cultural-names/blob/master/LICENSE.md). This notebook does not redistribute MCN data; it reads the repository in-place from `data/repos/more-cultural-names/`.

In [1]:
import sqlite3
import collections
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from dataclasses import dataclass
from enum import Enum

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT = Path.cwd().parent
MCN_PATH = PROJECT_ROOT / "data" / "repos" / "more-cultural-names"
MCN_LOCATIONS = MCN_PATH / "locations.xml"
DB_PATH = PROJECT_ROOT / "data" / "geonames.sqlite"
PACK_ID = "latin"

In [2]:
def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def run_query(sql: str, params: tuple = ()) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

In [3]:
# Get all MCN CK3 names

locations_tree = ET.parse(MCN_LOCATIONS)

ck3_names: dict[tuple[str, int], dict[str, str]] = {}
for loc in locations_tree.iter("LocationEntity"):
    geonames_tag = loc.find("GeoNamesId")
    if geonames_tag is None:
        continue

    geonames_id = int(geonames_tag.text)

    for game_id in loc.findall("GameIds/GameId"):
        if game_id.attrib.get("game") == "CK3":
            title = game_id.text.strip() if game_id.text is not None else ""

            if title.startswith("d_nf") or title.startswith("b_"):
                continue

            ck3_names[(title, geonames_id)] = {
                n.attrib["language"]: n.attrib["value"]
                for n in loc.findall("Names/Name")
            }

In [4]:
# Get all MCN Latin names that have a Geonames tag

ck3_latin_names: dict[tuple[str, int], str] = {}

for loc in ck3_names:
    names = ck3_names[loc]
    if "Latin" in names:
        ck3_latin_names[loc] = names["Latin"]
    else:
        ck3_latin_names[loc] = "MISSING"

In [5]:
ck3_latin_attested = {k: v for k, v in ck3_latin_names.items() if v != "MISSING"}
ck3_latin_missing  = {k for k, v in ck3_latin_names.items() if v == "MISSING"}

In [6]:
total   = len(ck3_latin_names)
attested = len(ck3_latin_attested)
print(f"CK3 titles with GeoNames ID : {total}")
print(f"  — Latin name attested (MCN): {attested} ({attested/total:.1%})")
print(f"  — MISSING                  : {total - attested} ({(total-attested)/total:.1%})")

CK3 titles with GeoNames ID : 218
  — Latin name attested (MCN): 118 (54.1%)
  — MISSING                  : 100 (45.9%)


In [7]:
class MatchStatus(Enum):
    EXACT        = "exact"       # MCN name appears verbatim in DB (any language tag)
    CASE_FOLD    = "case_fold"   # appears after lowercasing
    DB_EMPTY     = "db_empty"    # geonames_id has no alternate names at all
    DB_MISS      = "db_miss"     # alternate names exist, but MCN name not among them

@dataclass
class ComparisonResult:
    title:        str
    geonames_id:  int
    mcn_name:     str
    matched_lang: str | None     # isolanguage of the matching row, if found
    status:       MatchStatus

def compare_one(title: str, geonames_id: int, mcn_name: str) -> ComparisonResult:
    df = run_query("""
        SELECT isolanguage, alternate_name
        FROM alternate_name
        WHERE geonameid = ?
        ORDER BY is_preferred_name DESC, alternate_name
    """, (geonames_id,))

    if df.empty:
        return ComparisonResult(title, geonames_id, mcn_name, None, MatchStatus.DB_EMPTY)

    all_names = df["alternate_name"].tolist()

    # Exact match
    if mcn_name in all_names:
        lang = df.loc[df["alternate_name"] == mcn_name, "isolanguage"].iloc[0]
        return ComparisonResult(title, geonames_id, mcn_name, lang, MatchStatus.EXACT)

    # Case-insensitive match
    lower_map = {n.lower(): (n, row.isolanguage) 
                 for _, row in df.iterrows() 
                 for n in [row.alternate_name]}
    if mcn_name.lower() in lower_map:
        _, lang = lower_map[mcn_name.lower()]
        return ComparisonResult(title, geonames_id, mcn_name, lang, MatchStatus.CASE_FOLD)

    return ComparisonResult(title, geonames_id, mcn_name, None, MatchStatus.DB_MISS)

results = [
    compare_one(title, gid, mcn_name)
    for (title, gid), mcn_name in ck3_latin_attested.items()
]

counts = collections.Counter(r.status for r in results)
for status in MatchStatus:
    n = counts[status]
    print(f"{status.value:15s}  {n:4d}  ({n/len(results):.1%})")

exact              63  (53.4%)
case_fold           0  (0.0%)
db_empty            0  (0.0%)
db_miss            55  (46.6%)


In [8]:
# For titles MCN has no Latin name for, check what the DB does have

@dataclass
class MissingResult:
    title:       str
    geonames_id: int
    la_names:    list[str]
    untagged:    list[str]   # rows with empty isolanguage
    status:      str         # 'db_has_la' / 'db_has_untagged' / 'db_has_nothing'

def check_missing(title: str, geonames_id: int) -> MissingResult:
    df = run_query("""
        SELECT isolanguage, alternate_name
        FROM alternate_name
        WHERE geonameid = ?
          AND isolanguage IN ('la', '')
        ORDER BY isolanguage DESC, alternate_name
    """, (geonames_id,))

    la_names  = df.loc[df["isolanguage"] == "la",  "alternate_name"].tolist()
    untagged  = df.loc[df["isolanguage"] == "",     "alternate_name"].tolist()

    if la_names:
        status = "db_has_la"
    elif untagged:
        status = "db_has_untagged"
    else:
        status = "db_has_nothing"

    return MissingResult(title, geonames_id, la_names, untagged, status)

missing_results = [
    check_missing(title, gid)
    for (title, gid) in ck3_latin_missing
]

# Summary
miss_counts = collections.Counter(r.status for r in missing_results)
for status, n in miss_counts.most_common():
    print(f"{status:20s}  {n:4d}  ({n/len(missing_results):.1%})")

print()

db_has = [r for r in missing_results if r.status != "db_has_nothing"]
for r in sorted(db_has, key=lambda r: r.title):
    names = r.la_names or r.untagged
    tag   = "la" if r.la_names else "untagged"
    print(f"{r.title:35s}  [{tag}]  {names}")

db_has_untagged         75  (75.0%)
db_has_nothing          17  (17.0%)
db_has_la                8  (8.0%)

c_SUM_pekanbaru                      [untagged]  ['Pakanbahru', 'Pakanbaroe', 'Pakanbaru']
c_abomey                             [untagged]  ['Abome', 'Abomey']
c_al-gharbiya                        [untagged]  ['Al Gharbīyah']
c_al_aqabah                          [untagged]  ['Qal`at el `Aqaba', '`Aqaba']
c_aqtobe                             [untagged]  ['Aktiubinsk', 'Aktobe', 'Aktubinsk', 'Aktyubinsk', 'Ukhtiubinskii']
c_atkarsk                            [untagged]  ['Atarsk', 'Atkarsk']
c_baydhabo                           [untagged]  ['Baidabho', 'Baidabo', 'Iscia Baidoa', 'Isha Baydabo', 'Isha Baydhaba', 'Isha Baydhabo']
c_busaso                             [untagged]  ['Bandar Kassim', 'Bender Qaasin', 'Boosaso', 'Bosaso', 'Bosasso', 'Bénder Qāsin']
c_buzachi                            [untagged]  ['Bozashchy Tübegi', 'Buzachi Peninsula', 'Halbinsel Busatschi', 'Poluostrov 